---

## 5. Modelling

### 5.1 Membangun Algoritma K-Means++ Clustering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fungsi untuk menghitung jarak garis lurus (Euclidean Distance) antara dua titik
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

class KMeansPlusPlus:

    # Fungsi konstruktor yang otomatis berjalan pertama kali saat objek dibuat
    def __init__(self, K=3, max_iters=100, plot_steps=False, random_state=43):
        self.K = K
        self.max_iters = max_iters
        self.plot_steps = plot_steps
        self.random_state = random_state

        self.clusters = [[] for _ in range(self.K)]
        self.centroids = []
        self.labels = None

    # Fungsi inisialisasi centroid awal menggunakan strategi K-Means++
    def _init_centroids(self):
        if self.random_state is not None:
            np.random.seed(self.random_state)

        first_idx = np.random.randint(self.n_samples)
        centroids = [self.X[first_idx]]

        for _ in range(1, self.K):
            dist_sq = np.array([
                min([np.sum((x - c) ** 2) for c in centroids])
                for x in self.X
            ])

            sum_dist = dist_sq.sum()
            if sum_dist == 0:
                probs = np.ones(self.n_samples) / self.n_samples
            else:
                probs = dist_sq / sum_dist

            probs = probs / probs.sum()

            next_idx = np.random.choice(self.n_samples, p=probs)
            centroids.append(self.X[next_idx])

        return np.array(centroids)

    # Fungsi utama untuk memulai proses pengelompokan pada matriks fitur X
    def predict(self, X):
        self.X = np.nan_to_num(np.asarray(X, dtype=float))
        self.n_samples, self.n_features = self.X.shape

        self.centroids = self._init_centroids()

        for _ in range(self.max_iters):
            self.clusters = self._create_clusters(self.centroids)

            if self.plot_steps:
                self.plot()

            centroids_old = self.centroids
            self.centroids = self._get_centroids(self.clusters)

            if self._is_converged(centroids_old, self.centroids):
                break

            if self.plot_steps:
                self.plot()

        self.labels = self._get_cluster_labels(self.clusters)
        return self.labels

    # Fungsi untuk mengubah daftar indeks kelompok menjadi array label akhir yang urut
    def _get_cluster_labels(self, clusters):
        labels = np.empty(self.n_samples, dtype=int)
        for cluster_idx, cluster in enumerate(clusters):
            for sample_idx in cluster:
                labels[sample_idx] = cluster_idx
        return labels

    # Fungsi untuk mengelompokkan seluruh titik data ke dalam cluster dengan centroid terdekat
    def _create_clusters(self, centroids):
        clusters = [[] for _ in range(self.K)]
        for idx, sample in enumerate(self.X):
            centroid_idx = self._closest_centroid(sample, centroids)
            clusters[centroid_idx].append(idx)
        return clusters

    # Fungsi untuk mencari indeks centroid yang jaraknya paling dekat dengan satu sampel
    def _closest_centroid(self, sample, centroids):
        distances = [euclidean_distance(sample, point) for point in centroids]
        closest_idx = np.argmin(distances)
        return closest_idx

    # Fungsi untuk memperbarui posisi centroid ke nilai rata-rata kelompok terbaru
    def _get_centroids(self, clusters):
        centroids = np.zeros((self.K, self.n_features))
        for cluster_idx, cluster in enumerate(clusters):
            if len(cluster) > 0:
                cluster_mean = np.mean(self.X[cluster], axis=0)
                centroids[cluster_idx] = cluster_mean
            else:
                centroids[cluster_idx] = self.centroids[cluster_idx]
        return centroids

    # Fungsi untuk memeriksa apakah posisi centroid sudah tidak bergeser lagi
    def _is_converged(self, centroids_old, centroids):
        distances = [euclidean_distance(centroids_old[i], centroids[i]) for i in range(self.K)]
        return sum(distances) == 0

### 5.2 Menyiapkan Matrik Fitur dan Melatih Model

In [ ]:
kmeans_pp = KMeansPlusPlus(K=3, max_iters=100, plot_steps=False, random_state=43)
cluster_labels = kmeans_pp.predict(X_scaled)

print(" Ukuran Matriks Input :", X_scaled.shape)
print(" Bentuk Koordinat Centroid Akhir :", kmeans_pp.centroids.shape)
print(" Contoh 10 Label Cluster Pertama :", cluster_labels[:10])

### 5.3 Menggabungkan Label Cluster

In [ ]:
df_clustered = df.copy()
df_clustered['Cluster'] = cluster_labels

print("Jumlah Negara pada Setiap Cluster:")
print(df_clustered['Cluster'].value_counts().sort_index())

---

## 6. Evaluation

### 6.1 Menghitung Metrik Evaluasi Model (WCSS / SSE dan Silhouette Score)

In [ ]:
# Fungsi untuk menghitung Within-Cluster Sum of Squares (WCSS / SSE)
def hitung_wcss(X, labels, centroids):
    wcss = 0.0
    for i in range(len(centroids)):
        cluster_points = X[labels == i]
        if len(cluster_points) > 0:
            wcss += np.sum((cluster_points - centroids[i]) ** 2)
    return wcss

# Fungsi untuk menghitung Silhouette Score
def hitung_silhouette_score(X, labels):
    n_samples = len(X)
    unique_clusters = np.unique(labels)
    if len(unique_clusters) < 2:
        return 0.0

    dist_matrix = np.sqrt(np.sum((X[:, np.newaxis, :] - X[np.newaxis, :, :]) ** 2, axis=2))

    silhouette_vals = np.zeros(n_samples)
    for i in range(n_samples):
        own_cluster = labels[i]
        mask_own = (labels == own_cluster)
        mask_own[i] = False

        if np.sum(mask_own) > 0:
            a_i = np.mean(dist_matrix[i, mask_own])
        else:
            a_i = 0.0

        b_i = np.inf
        for other_cluster in unique_clusters:
            if other_cluster != own_cluster:
                mask_other = (labels == other_cluster)
                if np.sum(mask_other) > 0:
                    mean_dist_other = np.mean(dist_matrix[i, mask_other])
                    if mean_dist_other < b_i:
                        b_i = mean_dist_other

        denom = max(a_i, b_i)
        silhouette_vals[i] = (b_i - a_i) / denom if denom > 0 else 0.0

    return np.mean(silhouette_vals)

wcss_utama = hitung_wcss(X_scaled, cluster_labels, kmeans_pp.centroids)
silhouette_utama = hitung_silhouette_score(X_scaled, cluster_labels)

print(f"Hasil Evaluasi Model K-Means++ (K = {kmeans_pp.K}):")
print(f"- Nilai WCSS / SSE   : {wcss_utama:.4f}")
print(f"- Silhouette Score   : {silhouette_utama:.4f}")

### 6.2 Evaluasi Perbandingan Jumlah Cluster (K = 1 - 10)

In [ ]:
k_range = range(1, 10)
wcss_list = []
silhouette_list = []

for k in k_range:
    model_uji = KMeansPlusPlus(K=k, max_iters=100, plot_steps=False, random_state=43)
    labels_uji = model_uji.predict(X_scaled)
    
    wcss_val = hitung_wcss(X_scaled, labels_uji, model_uji.centroids)
    wcss_list.append(wcss_val)
    
    if k >= 2:
        sil_val = hitung_silhouette_score(X_scaled, labels_uji)
        silhouette_list.append(sil_val)
    else:
        silhouette_list.append(np.nan)

df_evaluasi_k = pd.DataFrame({'Jumlah Cluster (K)': list(k_range), 'WCSS / SSE': np.round(wcss_list, 4), 'Silhouette Score': np.round(silhouette_list, 4)})
print("Tabel Perbandingan Evaluasi Nilai K:")
display(df_evaluasi_k)

### 6.3 Evaluasi Profil Karakteristik Cluster & Pemetaan Kategori Pembangunan Negara

In [ ]:
fitur_evaluasi = ['child_mort', 'exports', 'health', 'imports', 'income', 'inflation', 'life_expec', 'total_fer', 'gdpp']

urutan_cluster_gdpp = df_clustered.groupby('Cluster')['gdpp'].mean().sort_values().index.tolist()

nama_kategori = ["Negara Prioritas Bantuan", "Negara Berkembang", "Negara Maju"]

cluster_label_map = {
    cluster_id: nama_kategori[idx] 
    for idx, cluster_id in enumerate(urutan_cluster_gdpp)
}

df_clustered['Kategori_Pembangunan'] = df_clustered['Cluster'].map(cluster_label_map)

print("Pemetaan Nomor Cluster ke Kategori Pembangunan:")
for cid, kategori in cluster_label_map.items():
    jumlah_negara = (df_clustered['Cluster'] == cid).sum()
    print(f"- Cluster {cid} ({jumlah_negara} negara) -> {kategori}")

profil_rata_rata = df_clustered.groupby(['Cluster', 'Kategori_Pembangunan'])[fitur_evaluasi].mean().round(2)
print("\nTabel Evaluasi Rata-Rata Indikator Sosio-Ekonomi & Kesehatan per Cluster:")
display(profil_rata_rata)